<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Implement Tool Error Handling _sol</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

#Purpose
Make your agent reliable in real-world conditions by handling tool failures gracefully.

In production, tools fail because of:

1. Network timeouts / connection errors (real external APIs)
2. Rate limits or service outages
3. Invalid inputs (e.g., division by zero, bad city name)
4. Empty retrieval results

You will:

Simulate realistic tool failures
Implement retry logic (with exponential backoff)
Add fallback behavior when retries fail
Observe full failure trace and recovery trace in the agent

This exercise builds directly on your previous “Build Three Agent Tools” exercise.

**What You Will Learn**

*   Why agents need explicit error handling
*   Real exceptions you will see in production (requests.exceptions.Timeout, ConnectionError, ZeroDivisionError, etc.)
*   How to use tenacity (the most popular retry library in LangChain agents)
*   Difference between retry (transient errors) and fallback (permanent failure)
*   How the ReAct agent sees failure traces and can recover
*   How to add logging so you can debug exactly what happened






#1. Setup

In [23]:
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.


In [24]:
!pip install sentence-transformers

In [25]:
!pip install tenacity

In [26]:
# Install dependencies

import os
import getpass
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from tenacity import retry, stop_after_attempt, wait_exponential_jitter, retry_if_exception_type
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done!")

Setup done!


#Real-World Tool Failure Scenarios

1. External API (get_weather) → network timeout, DNS failure, rate limit
2. Utility (calculate) → ZeroDivisionError, syntax error
3. Retrieval (retrieve_knowledge) → no matching documents or vectorstore issue

Without handling, the entire agent crashes or gives useless answers.
With retry + fallback → agent recovers automatically.

#Task 1: Simulate Realistic Failures (unreliable tools)

In [27]:
# === REAL ==
docs = [
    Document(page_content="The Eiffel Tower is a wrought iron lattice tower on the Champ de Mars in Paris, France. Designed by Gustave Eiffel, it was completed in 1889 and stands 324 metres tall. It receives over 7 million visitors per year.", metadata={"source": "landmarks"}),
    Document(page_content="Tesla, Inc. is an American multinational automotive and clean energy company headquartered in Austin, Texas. Founded in 2003... Elon Musk became CEO in 2008.", metadata={"source": "companies"}),
    Document(page_content="The Great Wall of China ... total length is over 21,000 km.", metadata={"source": "history"}),
    Document(page_content="The Burj Khalifa in Dubai is the tallest building in the world at 828 metres...", metadata={"source": "landmarks"}),
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Core functions (will be wrapped later)
def _get_weather_core(location: str) -> str:
    """Real weather with better error handling + Celsius"""
    try:
        # Using metric unit (Celsius) + simple format
        url = f"https://wttr.in/{location.replace(' ', '+')}?format=%C+%t+%w&unit=m"
        resp = requests.get(url, timeout=6)
        resp.raise_for_status()

        result = resp.text.strip()
        # Clean weird characters
        result = result.replace('Â', '').replace('°F', '°C').replace('â', '↓').replace('â', '←')

        return f"Weather in {location.title()}: {result}"

    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            raise ValueError(f"Location '{location}' not found")
        else:
            raise requests.exceptions.Timeout("Weather service returned error") from e
    except Exception as e:
        raise requests.exceptions.Timeout(f"Weather service unavailable: {type(e).__name__}") from e

def _calculate_core(expression: str) -> str:
    allowed = {"round": round, "abs": abs}
    return str(eval(expression, {"__builtins__": {}}, allowed))

def _retrieve_core(query: str) -> str:
    retrieved = retriever.invoke(query)
    if not retrieved:
        raise ValueError("No relevant documents found in knowledge base")
    return "\n\n".join([doc.page_content for doc in retrieved])

@tool
def unreliable_get_weather(location: str) -> str:
    """Real weather tool – fails realistically"""
    if "Fail" in location or "fail" in location:
        raise requests.exceptions.Timeout("Simulated real network timeout (like real API outage)")
    return _get_weather_core(location)

@tool
def unreliable_calculate(expression: str) -> str:
    """Math tool – fails on real math errors"""
    if "/0" in expression.replace(" ", ""):
        raise ZeroDivisionError("division by zero")
    return _calculate_core(expression)

@tool
def unreliable_retrieve_knowledge(query: str) -> str:
    """Retrieval tool – fails when no data"""
    if "Mars" in query or "capital of Mars" in query.lower():
        raise ValueError("No relevant documents found in knowledge base")
    return _retrieve_core(query)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#Task 2: Implement Retry + Fallback Wrapper

In [28]:

from functools import wraps

def robust_tool(fallback_message: str = "Service temporarily unavailable."):
    def decorator(func):
        @retry(
            stop=stop_after_attempt(3),
            wait=wait_exponential_jitter(initial=0.5, max=2.0),
            retry=retry_if_exception_type((requests.exceptions.Timeout,
                                           requests.exceptions.ConnectionError,
                                           ValueError)),  # add more as needed
            reraise=True
        )
        @wraps(func)
        def wrapper(*args, **kwargs):
            try:
                print(f"🔧 [TOOL START] {func.__name__} called with {args[0] if args else kwargs}")
                result = func(*args, **kwargs)
                print(f" [TOOL SUCCESS] {func.__name__}")
                return result
            except Exception as e:
                print(f" [TOOL ATTEMPT FAILED] {func.__name__} – {type(e).__name__}: {e}")
                raise  # let tenacity retry
        return wrapper
    return decorator


In [29]:
def robust_tool(fallback_message: str = "Service temporarily unavailable."):
    def decorator(func):
        @retry(
            stop=stop_after_attempt(3),
            wait=wait_exponential_jitter(initial=0.5, max=2.0),
            retry=retry_if_exception_type((requests.exceptions.Timeout,
                                           requests.exceptions.ConnectionError,
                                           ValueError,
                                           ZeroDivisionError)),
            reraise=True
        )
        @wraps(func)
        def wrapper(*args, **kwargs):
            try:
                print(f" [TOOL START] {func.__name__} called with {args[0] if args else kwargs}")
                result = func(*args, **kwargs)
                print(f"[TOOL SUCCESS] {func.__name__}")
                return result
            except Exception as e:   # final failure after retries
                print(f"💥 [FINAL FAILURE] {func.__name__} – all retries exhausted")
                print(f"   → Fallback activated: {fallback_message}")
                return f"{fallback_message} (original error: {type(e).__name__})"
        return wrapper
    return decorator

# ROBUST TOOLS
@tool
@robust_tool(fallback_message="Weather service temporarily unavailable.")
def get_weather(location: str) -> str:
    """Real weather with retry + fallback."""
    if "Fail" in location:
        raise requests.exceptions.Timeout("Simulated real network timeout")
    return _get_weather_core(location)

@tool
@robust_tool(fallback_message="Could not perform calculation.")
def calculate(expression: str) -> str:
    """Math with retry + fallback."""
    if "/0" in expression:
        raise ZeroDivisionError("division by zero")
    return _calculate_core(expression)

@tool
@robust_tool(fallback_message="No knowledge available at the moment.")
def retrieve_knowledge(query: str) -> str:
    """Retrieval with retry + fallback."""
    if "Mars" in query:
        raise ValueError("No relevant documents found")
    return _retrieve_core(query)

#Task 4: Build Two Agents

In [31]:
!pip install -q -U langsmith

from langsmith import Client

client = Client()
basic_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.0/738.0 kB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.


In [32]:
from langchain_classic.agents import create_react_agent, AgentExecutor


In [34]:
basic_tools = [unreliable_get_weather, unreliable_calculate, unreliable_retrieve_knowledge]
basic_agent = create_react_agent(llm, basic_tools, basic_prompt)
basic_executor = AgentExecutor(agent=basic_agent, tools=basic_tools, verbose=True, handle_parsing_errors=True)

In [36]:
robust_tools = [get_weather, calculate, retrieve_knowledge]
robust_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
robust_agent = create_react_agent(llm, robust_tools, robust_prompt)
robust_executor = AgentExecutor(agent=robust_agent, tools=robust_tools, verbose=True, handle_parsing_errors=True)

#Test Prompts (Validation)

In [37]:
test_prompts = [
    "What is the weather in Paris right now?",           # should succeed
    "What is the weather in FailParis?",                 # → triggers simulated timeout
    "Calculate 100 / 0",                                 # → ZeroDivisionError
    "Tell me about the capital of Mars",                 # → retrieval failure
    "Weather in Tokyo and calculate 25 * 4",             # mix of success + math
]

print("=== BASIC AGENT (expect failures) ===")
for p in test_prompts:
    print(f"\nQUERY: {p}")
    try:
        result = basic_executor.invoke({"input": p})
        print("ANSWER:", result["output"])
    except Exception as e:
        print("CRASHED:", type(e).__name__)

print("\n=== ROBUST AGENT (expect recovery) ===")
for p in test_prompts:
    print(f"\nQUERY: {p}")
    result = robust_executor.invoke({"input": p})
    print("ANSWER:", result["output"])

=== BASIC AGENT (expect failures) ===

QUERY: What is the weather in Paris right now?


> Entering new AgentExecutor chain...
I need to check the current weather in Paris. I'll use the weather tool to get this information. However, I should keep in mind that the tool may not always provide accurate results.  
Action: unreliable_get_weather  
Action Input: "Paris"  CRASHED: Timeout

QUERY: What is the weather in FailParis?


> Entering new AgentExecutor chain...
I need to check the weather in FailParis using the weather tool, but I should be aware that it may not always provide accurate results. 
Action: unreliable_get_weather
Action Input: "FailParis"CRASHED: Timeout

QUERY: Calculate 100 / 0


> Entering new AgentExecutor chain...
Dividing by zero is mathematically undefined, so I should use the unreliable_calculate tool to see how it handles this expression. 
Action: unreliable_calculate
Action Input: "100 / 0"CRASHED: ZeroDivisionError

QUERY: Tell me about the capital of Mars


> E